In [ ]:

!pip install -q transformers peft bitsandbytes accelerate datasets torch

In [ ]:
import random
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

In [ ]:
def generate_raft_dataset(num_examples: int = 50, p_golden_fraction: float = 0.8):
    questions = {
        "Explain Newton's first law of motion.": {
            "text": "Newton's first law states that an object remains at rest or in uniform motion unless acted upon by an external force.",
            "answer": "An object resists changes in motion unless acted upon by an external force.",
            "reason": "It describes inertia."
        },
        "What are the main differences between supervised and unsupervised learning?": {
            "text": "Supervised learning uses labeled data, while unsupervised learning identifies patterns in unlabeled data.",
            "answer": "Supervised uses labels, unsupervised does not.",
            "reason": "Targets vs. no targets."
        },
        "Why does the Earth experience seasons?": {
            "text": "The tilt of Earth's axis causes varying sunlight angles, leading to seasons.",
            "answer": "Seasons occur because of Earth's axial tilt.",
            "reason": "Different tilt angles change sunlight intensity."
        }
    }

    distractors = [
        "Einstein proposed the theory of relativity.",
        "The moon orbits Earth in about 27 days.",
        "Enzymes speed up chemical reactions."
    ]

    dataset = []
    for _ in range(num_examples):
        q, golden = random.choice(list(questions.items()))
        docs = random.sample(distractors, k=2)
        if random.random() < p_golden_fraction:
            docs.append(golden["text"])
            random.shuffle(docs)

        context = "\n".join(f"[Doc {i+1}: {d}]" for i, d in enumerate(docs))
        inp = f"Question: {q}\nContext:\n{context}\nInstruction: Answer with reasoning and final answer."
        out = f"##Reason: {golden['reason']}\n##Answer: {golden['answer']}"
        dataset.append({"input": inp, "output": out})

    return Dataset.from_list(dataset)

train_dataset = generate_raft_dataset(60)


In [ ]:


MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)


In [ ]:

def tokenize_fn(batch):
    full_texts = [f"{i}{o}{tokenizer.eos_token}" for i, o in zip(batch["input"], batch["output"])]
    tok = tokenizer(full_texts, truncation=True, max_length=512, padding="max_length")
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["input","output"])


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [ ]:

training_args = TrainingArguments(
    output_dir="./sft_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    optim="paged_adamw_8bit",      # Memory-efficient optimizer
    fp16=True,                     # Enable mixed-precision training
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("\n Starting RAFT fine-tuning...")
trainer.train()
print(" Training complete!")

/tmp/ipython-input-1295885991.py:13: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.



🚀 Starting RAFT fine-tuning...


Step,Training Loss
10,13.062000
20,2.122200


✅ Training complete!
